# Args & Kwargs — Session 03: Flexible Function Signatures

Real functions rarely know in advance exactly how many arguments callers will supply. `*args` and `**kwargs` solve that problem: they let a function accept any number of extra positional or keyword arguments and collect them into a tuple or a dictionary. This notebook opens the final topic of Section 03.

**The big picture.** A function signature is a contract about *how* arguments are matched to parameters. Positional arguments match by order, keyword arguments match by name, and defaults make some parameters optional. `*args` and `**kwargs` extend the contract to say and anything else that arrives.

**Why it matters.** These tools power flexible APIs: `print` itself accepts unlimited items, many libraries accept optional configuration flags, and well-designed public functions absorb future options without breaking existing callers.

**Roadmap.** In this notebook you will:
1. Compare positional and keyword calling styles and their error cases.
2. Use `*args` to pack extra positional arguments into a tuple.
3. Prove that `args` is always a tuple, even when empty.
4. See why `print` is the `*args` superstar and how to forward arguments.
5. Learn the signature order laws, including keyword-only parameters.

**Key definitions.** *Positional argument*: matched by position. *Keyword argument*: matched by `name=value`. *Default*: the value used when an argument is omitted. *`*args`*: a tuple of extra positionals. *Keyword-only*: parameters after a bare `*` that must be passed by name.

## 1. Positional vs Keyword Arguments

Python matches arguments to parameters in two ways. **Positional** arguments are matched by order: the first argument goes to the first parameter, and so on. **Keyword** arguments are matched by name: `city='Jaipur'` goes to the `city` parameter regardless of position.

Defaults make parameters optional. In `def introduce(name, city, hobby='coding')`, `hobby` may be omitted. The rules are strict:

- Positional arguments must come before keyword arguments in a call.
- A required parameter cannot follow one with a default in a definition.
- Missing required arguments raise `TypeError`.

```text
f(1, 2, x=3)   # valid: positionals first, then keywords
f(x=3, 1)      # invalid: positional after keyword
```

These rules exist so that every call can be resolved unambiguously. The experiments below show the successful styles and the missing-argument error.

In [1]:
# 🐣 Positional vs Keyword calls — style-boxing!
def introduce(name, city, hobby="coding"):
    print(f"I am {name} from {city}. I love {hobby}.")

introduce("Sana", "Delhi")                          # positional + default
introduce("Ravi", "Pune", "cricket")                # 3 positional
introduce(name="Meena", hobby="dance", city="Jaipur")  # pure-key (order-free!)
try:
    introduce("Kavi")                               # city missing!
except TypeError as e:
    print("missing-arg 🚨:", e)
# Expected OUTPUT:
# I am Sana from Delhi. I love coding.
# I am Ravi from Pune. I love cricket.
# I am Meena from Jaipur. I love dance.
# missing-arg 🚨: introduce() missing 1 required positional argument: 'city'

I am Sana from Delhi. I love coding.
I am Ravi from Pune. I love cricket.
I am Meena from Jaipur. I love dance.
missing-arg 🚨: introduce() missing 1 required positional argument: 'city'


## 2. *args — The Flexible Bag

Placing `*args` in a signature tells Python to collect any extra positional arguments into a tuple called `args`. The name `args` is only a convention; any name after the star works, but `args` is idiomatic.

`def order_summary(customer, *items)` fixes one required parameter and accepts zero or more items after it. Whether the caller passes one item or ten, the function receives a tuple and can loop over it, take its length, or sum it.

This is the flexible-bag idea: the caller packs as many items as needed, and the function unpacks the bag with ordinary tuple operations. It is how libraries build APIs that grow without breaking old calls.

In [2]:
# 🐥 `*args` — flexible thali! 🛍️ (chemist-window live!)
def order_summary(customer, *items):
    print(f"{customer} ordered {len(items)} items:")
    for i, item in enumerate(items, start=1):
        print(f"  #{i}: {item}")

order_summary("Sana", "paracetamol")
print()
order_summary("Ravi", "bread", "milk", "eggs", "butter")
# Expected OUTPUT:
# Sana ordered 1 items:
#   #1: paracetamol
#
# Ravi ordered 4 items:
#   #1: bread
#   #2: milk
#   #3: eggs
#   #4: butter

Sana ordered 1 items:
  #1: paracetamol

Ravi ordered 4 items:
  #1: bread
  #2: milk
  #3: eggs
  #4: butter


## 3. *args Is Always a Tuple

Whatever the caller passes, `args` is a tuple — never a list. That matters because tuples are immutable: you cannot append to `args`, and you should not try. When you need a list, convert explicitly.

| Call | `args` value | Type |
|---|---|---|
| `f(10, 20, 30)` | `(10, 20, 30)` | tuple |
| `f(7)` | `(7,)` | tuple |
| `f()` | `()` | tuple |

Because a tuple supports `len`, iteration, and functions like `sum`, `args` is convenient for building helpers such as an average that safely handles the empty case. The empty tuple is a normal input, so guard divisions by zero when it appears.

In [3]:
# 🐥 args har-case TUPLE hai — proof + stats helper
def avg_of(*nums):
    print(f"args type: {type(nums).__name__} | value: {nums}")
    if not nums:
        return 0
    return sum(nums) / len(nums)

print("avg:", avg_of(10, 20, 30))
print("avg:", avg_of(7))
print("avg:", avg_of(), "← empty safe ✅")
# Expected OUTPUT:
# args type: tuple | value: (10, 20, 30)
# avg: 20.0
# args type: tuple | value: (7,)
# avg: 7.0
# args type: tuple | value: ()
# avg: 0 ← empty safe ✅

args type: tuple | value: (10, 20, 30)
avg: 20.0
args type: tuple | value: (7,)
avg: 7.0
args type: tuple | value: ()
avg: 0 ← empty safe ✅


## 4. print() — The *args Superstar

`print` accepts any number of items, which is exactly `*args` in action: `print(a, b, c)`. Its `sep` and `end` options are keyword arguments, so the full signature is roughly `print(*objects, sep=' ', end='\n')`.

A useful pattern is **forwarding**: define `def wrapper(*args): return print('FORWARDED:', *args)`. The star collects the extra positionals, and a second star at the call site unpacks them into `print`. This is how decorators and thin wrappers pass arguments through unchanged while adding behaviour.

Remember the star is symmetric: at a definition it packs, and at a call site it unpacks.

In [4]:
# 🐔 print() khud *args ka GOAT 🐐 — siphai example!
print("print", "takes", "unlimited", "items", "(sep=' ' default)")
print("a", "b", "c", sep=" - ", end="!\n")         # sep/end = print ke kwargs!

def private_forward(*args):
    return print("FORWARDED:", *args)              # args agle pe pass — star uthta!
private_forward(1, 2, 3)
# Expected OUTPUT:
# print takes unlimited items (sep=' ' default)
# a - b - c!
# FORWARDED: 1 2 3

print takes unlimited items (sep=' ' default)
a - b - c!
FORWARDED: 1 2 3


## 5. Signature Anatomy — The Order Laws

Python fixes the order of parameter kinds. A well-formed signature reads:

```python
def mega(pos1, pos2,        # 1) required positionals
         name='default',    # 2) parameters with defaults
         *args,             # 3) extra positionals, packed as a tuple
         kw_only=0,         # 4) keyword-only parameters (after the star)
         **kwargs):         # 5) extra keywords, packed as a dict
    ...
```

Call-side rules mirror this: positionals first, then keywords. Definition-side rules: required parameters precede defaults, `*args` precedes keyword-only parameters, and `**kwargs` comes last.

Anything placed after the bare `*`, or after `*args`, can only be passed by keyword. This creates a natural strict zone for optional flags. Used well, keyword-only parameters make intent explicit and make signatures safe to extend later.

## 5.1 Keyword-Only Arguments

A bare `*` in a signature means everything after this must be passed by name. In `def make_bill(item, price, *, discount=0, delivery=0)`, the discount and delivery cannot be supplied positionally.

This is deliberate API design. Keyword-only flags are self-documenting at the call site (`discount=0.1` reads better than a mystery positional), impossible to confuse by order, and forward-compatible: adding a new keyword-only option never shifts the meaning of existing positional calls.

Passing such an argument positionally raises `TypeError`. The experiment below shows the error message, which confirms the strict zone.

In [5]:
# 🐔 Keyword-ONLY demo — `*` ke baad wale strict-honest!
def make_bill(item, price, *, discount=0, delivery=0):
    return price - price * discount + delivery

print(make_bill("Laptop", 48000))
print(make_bill("Laptop", 48000, discount=0.1, delivery=500))
try:
    make_bill("Laptop", 48000, 0.1)          # kw-only ko positional se nahi de sakte!
except TypeError as e:
    print("kw-only positional 🚨:", e)
# Expected OUTPUT:
# 48000.0
# 43700.0
# kw-only positional 🚨: make_bill() takes 2 positional arguments but 3 were given

48000
43700.0
kw-only positional 🚨: make_bill() takes 2 positional arguments but 3 were given


## ⚠️ 5 Common Mistakes (Args Special)

| # | ❌ Mistake | ✅ Correct |
|---|---|---|
| 1 | `f(x=1, 2)` positional after keyword | positional first, keyword after! (`f(2, x=1)`) |
| 2 | `def f(a=1, b)` default-then-required | required first: `def f(b, a=1)` |
| 3 | Expecting `*args` to be a list (trying to mutate!) | args is an immutable TUPLE — need a list? convert |
| 4 | A positional call for kw-only | name=value is required post-`*` 📝 |
| 5 | Mutable defaults! (`= []` flashback 🪤) | `= None` + guard (remember 03.04!) |

> 🎯 **Design tip:** use kw-only (`*, flag=...`) in public APIs — callers stay
> EXPLICIT, fewer bugs; future-proof signatures! ✨

## Summary

- Positional arguments match by order and must precede keyword arguments in a call.
- Defaults make parameters optional; a required parameter cannot follow a default.
- `*args` packs extra positional arguments into an immutable tuple, even when empty.
- `print` is the classic `*args` function, and forwarding with `*args` passes arguments through unchanged.
- The signature order is positional, defaults, `*args`, keyword-only, `**kwargs`.
- Anything after a bare `*` is keyword-only and must be passed by name.